# Stage 33A: fold-safe TCN cut-benefit gate

固定済みStage 30 TCNからfold-safe training予測を作り、cutごとの補正係数を学習します。TCNの再学習、予約120 wellsの使用、Kaggle提出作成は行いません。GPUランタイムを選択してください。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
data_dir = drive_root / 'data'

if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before running Stage 33A'
print(torch.cuda.get_device_name(0))


In [ ]:
stage16b_run = artifact_dir / 'stage16b_testlike_validation_full_v003'
stage17a_run = artifact_dir / 'stage17_public_replay_full_v002'
public_oof_run = artifact_dir / 'stage7_public_residual_gate_full_v001'
stage21b_run = artifact_dir / 'stage21b_prefix_confidence_full_v001'
stage30a_run = artifact_dir / 'stage30a_residual_tcn_full_v001'
manifest_run = artifact_dir / 'stage29a_multicut_manifest_v001'
required = [
    stage16b_run / 'well_assignments.parquet',
    stage17a_run / 'cut_report.parquet',
    public_oof_run / 'base_oof.parquet',
    stage21b_run / 'confidence_cut_report.parquet',
    manifest_run / 'training_cut_ids.parquet',
    manifest_run / 'confirmation_cut_ids.parquet',
    stage30a_run / 'summary.json',
    stage30a_run / 'normalizer.npz',
] + [stage30a_run / f'fold_{fold}.pt' for fold in range(5)]
for path in required:
    assert path.is_file(), path
manifest = json.loads((manifest_run / 'summary.json').read_text())
assert manifest['training_wells'] == 500 and manifest['confirmation_wells'] == 120
assert manifest['reserved_confirmation_used'] is False
print(manifest['training_cuts'], 'fold-safe training cuts; reserve remains sealed')


In [ ]:
RUN_ID = 'stage33a_tcn_cut_benefit_gate_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'summary.json').is_file():
    command = [
        sys.executable, '-m', 'rogii.cli.residual_cut_benefit_gate',
        '--config', 'configs/experiment/stage33a_tcn_cut_benefit_gate.yaml',
        '--stage16b-run', str(stage16b_run),
        '--stage17a-run', str(stage17a_run),
        '--public-oof-run', str(public_oof_run),
        '--stage30a-run', str(stage30a_run),
        '--split-manifest-run', str(manifest_run),
        '--validation-run', str(stage21b_run),
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID,
        '--device', 'cuda',
    ]
    run_env = os.environ.copy()
    run_env['PYTHONPATH'] = str(repo_dir / 'src') + ':' + run_env.get('PYTHONPATH', '')
    result = subprocess.run(command, cwd=repo_dir, env=run_env, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError(f'Stage 33A failed with exit code {result.returncode}')
else:
    print('Reusing completed run:', run_dir)


In [ ]:
import pandas as pd

summary = json.loads((run_dir / 'summary.json').read_text())
display(pd.DataFrame(summary['family_reports']['stage16_fold']['fold_report']))
{key: summary[key] for key in [
    'stage33a_complete',
    'promoted_to_stage33b_reserved_confirmation',
    'device',
    'training_cuts',
    'training_wells',
    'validation_cuts',
    'validation_wells',
    'gate_feature_count',
    'training_crossfit_gate_mae',
    'validation_predicted_gate_mean',
    'validation_oracle_gate_mean',
    'base_rmse',
    'candidate_rmse',
    'rmse_delta',
    'bootstrap_95pct',
    'cut_rmse_p90_delta',
    'gates',
    'reserved_confirmation_used',
    'next_step',
]}
